In [ ]:
# Purpose - analyze only the midline data

fis = [63, 23] #[23, 63] #[26, 63] #[23, 63] #63 #6 #25

In [ ]:
# Imports and Folder Selection

import sys

import numpy as np
import pandas as pd
import xarray as xr

import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from graphet import Data
from graphet.plugins import TristanV2 

sys.path.append('..')
from useful_funcs import *
sys.path.append('../fmgmt')
from TristanRun import TristanRun
#from FieldLineHelper import FieldLineHelper

plt.close('all')
%matplotlib widget
set_standard_plot_params()

# Read Input File Data

runs = [TristanRun(findex) for findex in fis]

print(runs[1].sigma)

In [ ]:
# Configure Data

midlines = [xr.open_dataset(f"{TristanRun.find_folderpath(findex)}/output/cut_flds/midline.hdf5", engine='h5netcdf') for findex in fis]

durations = [len(midlines.t) * run.beta_A for midlines, run in zip(midlines, runs)]
print(f"Before: {durations}")

min_dur = min(durations)

midlines = [midline.isel(t=slice(0, int(min_dur / run.beta_A))) for midline, run in zip(midlines, runs)]

durations = [len(midlines.t) * run.beta_A for midlines, run in zip(midlines, runs)]
print(f"After: {durations}")

In [ ]:
counter = 0
run_index = 1
x1, x2 = {10: [150, 200], 0.3: [110, 130]}[runs[run_index].sigma]
#x1, x2 = {10: [160, 175], 0.3: [117.5, 122.5]}[runs[run_index].sigma]

for ts in range(400):
    plt.figure()
    plt.close('all')
    run = runs[run_index]
    
    j0 = run.cc * run.B0 / run.c_omp 

    midline = midlines[run_index].isel(t = ts).sel(y = slice(x1, x2))

    midline['ez_ni'] = midline.ez + midline.velx * midline.by - midline.vely * midline.bx 
    midline['ex_ni'] = midline.ex + midline.vely * midline.bz - midline.velz * midline.by
    midline['ey_ni'] = midline.ey + midline.velz * midline.bx - midline.velx * midline.bz
    midline['eta'] = (midline.ez_ni / run.B0) / (midline.jz / j0)

    #plt.plot(midline.y, midline.dens, label='$n / n_0$')
    
    #plt.plot(midline.y, midline.dens, label='$n$', color = 'blue')
    plt.plot(midline.y, 5 * midline.bx / (run.B0), label='$5 B_x$', color = 'green')
    #plt.plot(midline.y, 10 * midline.ex_ni / (run.B0), label='$10 E_xni$', color = 'green')
    #plt.plot(midline.y, 10 * midline.ey_ni / (run.B0), label='$10 E_yni$')
    #plt.plot(midline.y, 10 * midline.ez_ni / (run.B0), label='$10 E_zni$')

    #plt.plot(midline.y, midline.bx / (run.B0), label='$10 B_x$')
    #plt.plot(midline.y, midline.by / (run.B0), label='$10 B_y$')
    #plt.plot(midline.y, midline.bz / (run.B0), label='$10 B_z$')

    adj = {10: 150, 0.3: 5}[run.sigma]

    #plt.plot(midline.y, 10 * (midline.jz / run.J0) * (midline.ez / run.B0), label = 'J dot E', color = 'violet')
    #plt.plot(midline.y, adj * (midline.jz / run.J0) ** 2 / (midline.epsilon * midline.dens), label = f'{adj} $j^2 / (n \epsilon)$', color = 'red')
    #plt.plot(midline.y, (midline.jz / run.J0) / midline.dens, label = 'j / n', color = 'violet')
    
    plt.plot(midline.y, 5 * midline.jz / run.J0, label='$5 j_z / j_0$', color = 'orange')

    #plt.plot(midline.y, midline.ez / (0.1 * run.beta_A * run.B0), label='$E_z / (0.1 v_A B_0) $', color = 'cyan')
    
    #plt.plot(midline.y, 10 * midline.eta , label='$10 \eta$', color = 'purple')
    plt.plot(midline.y, 5 * midline.vely, label='$5 v_y$', color = 'purple')
    #plt.plot(midline.y, (midline.jz / j0) / midline.dens, label='$j / n$', color = 'orange')
    #plt.plot(midline.y, midline.epsilon - 1, label='$\epsilon - 1$', color = 'green')

    plt.axhline(y=0, color='k', linestyle=':', linewidth=1)


    # Shade null points (x-points)
    threshold = 0.1  # set your threshold here
    btot = np.sqrt(midline.bx ** 2 + midline.by ** 2 + midline.bz ** 2) / run.B0
    y = midline.y.values

    below = btot < threshold
    in_region = False
    start = None

    for idx, val in enumerate(below):
        if val and not in_region:
            start = y[idx]
            in_region = True
        elif not val and in_region:
            end = y[idx]
            plt.axvspan(start, end, color='red', alpha=0.2)
            in_region = False

    # Plot vertical red lines where vy changes sign
    vy = midline.vely.values
    sign_changes = np.where(np.diff(np.sign(vy)) != 0)[0]
    for idx in sign_changes:
        plt.axvline(x=y[idx], color='red', linestyle='-', linewidth=1.5, alpha=0.7)
    

    # Handle case where region goes to end
    if in_region:
        plt.axvspan(start, y[-1], color='red', alpha=0.2)

    plt.ylim([-1.5, 5])

    

    plt.title(f'sigma = {run.sigma}, findex = {fis[run_index]} \n ts = {run.steps[ts]}, $\Omega t$ = {ts * run.gyrofreq:.02f}')
    plt.legend()
    save_movie_frame(counter)
    counter += 1

plt.show()

In [ ]:
from scipy.stats import expon, kstest
import seaborn as sns

plasmoid_arrs = {
    0.3: [2355, 2388, 2490, 2518, 2613, 2686, 2703, 2703, 2473, 2765, 2782, 2906, 2968, 3013], 
    10: [825, 825, 825, 825, 850, 858, 885, 906, 921, 953]
}


plt.figure(figsize=(8, 3))
plt.plot(plasmoid_arrs[0.3], np.full_like(plasmoid_arrs[0.3], 0.3, dtype=float), 'o', label='$\sigma$ = 0.3')
plt.xlabel('Timestep')
plt.ylabel('$\sigma$')
plt.title('Plasmoid Timesteps for $\sigma=0.3$')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 3))
plt.plot(plasmoid_arrs[10], np.full_like(plasmoid_arrs[10], 10, dtype=float), 'o', label='$\sigma$ = 10')
plt.xlabel('Timestep')
plt.ylabel('$\sigma$')
plt.title('Plasmoid Timesteps for $\sigma=10$')
plt.legend()
plt.tight_layout()
plt.show()

# Compare plasmoid time intervals to Poisson (exponential) process
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
sigmas = [0.3, 10]

axes[0].set_ylabel('Density')
plt.suptitle('Plasmoid Interval Distribution vs Poisson Process')
plt.tight_layout(rect=[0, 0, 1, 0.l95])
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(midlines[0].y, midlines[0].vely.isel(t=0), label='vely (t=0)')
plt.xlabel('$y$ [$c / \omega_p$]')
plt.ylabel('$v_{ey}$')
plt.title('Electron $v_y$ at Initial Timestep')
plt.legend()
plt.show()

In [ ]:
# Compute and plot the average FFT amplitude of vely for each timestep in midlines[0] and midlines[1]

plt.figure(figsize=(8, 4))

for i in range(2):
    n_timesteps = midlines[i].bx.shape[0]
    fft_amps = []
    for t in range(n_timesteps):
        var_t = midlines[i].ez.isel(t=t).values
        fft_vals = np.fft.fft(var_t)
        fft_amp = np.abs(fft_vals)
        fft_amps.append(fft_amp)
    fft_amps = np.array(fft_amps)
    avg_fft = np.mean(fft_amps, axis=0)
    freqs = np.fft.fftfreq(len(var_t), d=(y[1] - y[0]))
    plt.loglog(freqs[:len(freqs)//2], avg_fft[:len(avg_fft)//2], label=f"$\sigma$ = {runs[i].sigma}")

plt.xlabel('Wavenumber $k$ [1/$c \omega_p$]')
plt.ylabel('Amplitude')
plt.title('Mean FFT')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
print(midlines[1])

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), sharey = True)
plt.subplots_adjust(wspace=0)

fig_labels = ["(a)", "(b)"]


for i in range(2):
    run = runs[i]
    midline = midlines[i]
    ax = [ax1, ax2][i]

    xa, ya = [0.02, 0.99]

    ax.text(xa, ya, f"$\sigma$ = {run.sigma}", color='black', ha='left', va='top',
            transform=ax.transAxes, fontsize = 20, weight = "bold",
            bbox=dict(boxstyle='square,pad=0.3', edgecolor='white', facecolor='white', alpha=0.8))

    t_max = run.beta_A * run.cc / run.c_omp * run.output_interval * len(midline.t)
    #t_max = len(midline.t)

    im = ax.imshow(np.flipud(midline.dens), aspect='auto', extent=[midline.y.min(), midline.y.max(), 0, t_max], 
                   vmin=0, vmax=10)
    #fig.colorbar(im, ax=ax1, label='$n / n_0$')
    ax.set_xlabel('$y$ [$c / \omega_p$]')
    if (i == 0):
        ax.set_ylabel('$v_A t$ [$c / \omega_p$]')
    #ax.set_title(f"{fis[i]} sigma = {run.sigma}")

    ax.text(0.02, 0.02, fig_labels[i], color='white', ha='left', va='bottom',
        transform=ax.transAxes, fontsize=18, weight="bold",
        bbox=dict(boxstyle='square,pad=0.3', edgecolor='none', facecolor='black', alpha=0.5))
    
cbar = fig.colorbar(im, ax=[ax1, ax2], location='right', fraction=0.046, pad=0.04)
cbar.set_label('$n / n_0$')

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(np.flipud(midlines[1].dens), aspect='auto', vmin=0, vmax=10)
plt.axis('off')
plt.xticks([])
plt.yticks([])
plt.show()

In [ ]:
midlines['edens'] = midlines.dens * (midlines.epsilon - 1)

plt.figure(figsize=(10, 6))
t_max = (d.fields.t.max() - d.fields.t.min()) * run.gyrofreq
plt.imshow(np.flipud(midlines.edens), aspect='auto', extent=[d.fields.y.min(), d.fields.y.max(), 0, t_max], vmin = 0, vmax = 20)
plt.colorbar(label='KE Density')
plt.xlabel('$x$ [$c / \omega_p$]')
plt.ylabel('$\Omega t$')
plt.title(boilerplate)
plt.show()